In [ ]:
import MeshFEM
import mesh, mesh_energy
import numpy as np
import dirichlet_demo

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh')

In [ ]:
import parametrization
optVars = mesh_energy.NodalVars(m, 2)
optVars.setVars(parametrization.lscm(m).ravel())

In [ ]:
de_ad = dirichlet_demo.param_dirichlet_edensity_ad(m, optVars)
de = dirichlet_demo.param_dirichlet_edensity(m, optVars)
de_elem = dirichlet_demo.param_dirichlet_element(m, optVars)
de_elem_ad = dirichlet_demo.param_dirichlet_element_ad(m, optVars)
sde_elem_ad = dirichlet_demo.param_symdirichlet_element_ad(m, optVars)

In [ ]:
# Finite difference validations of gradient and Hessian
import fd_validation, py_newton_optimizer
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(optVars, [de, de_ad, de_elem, de_elem_ad])

fd_validation.gradConvergencePlot(prob)
fd_validation.hessConvergencePlot(prob)

In [ ]:
# Compare against Dirichlet energy calculated using cotan Laplacian matrix.
import differential_operators
L = differential_operators.laplacian(m)
u = optVars.getVars().reshape((-1, 2))[:, 0]
v = optVars.getVars().reshape((-1, 2))[:, 1]

0.5 * (u.dot(L.apply(u)) + v.dot(L.apply(v)))

### Compare the `SymDirichletParamElementAD` autodiff element against the `SymmetricDirichletDerivativeFree` autodiff energy density

In [ ]:
import energy, benchmark
param_sd = mesh_energy.Parametrization(m, optVars, energy.SymmetricDirichletDerivativeFree(2))
param_sd_an = mesh_energy.Parametrization(m, optVars, energy.SymmetricDirichlet(2))

In [ ]:
np.linalg.norm(param_sd.gradient() - sde_elem_ad.gradient())

In [ ]:
np.linalg.norm(param_sd.hessian().H_ss.Ax - sde_elem_ad.hessian().H_ss.Ax)

In [ ]:
# Time the x-based-autodiff element derivative evaluation.
# Note that `AutodiffElement` computes and caches per-element Hessian on each variable
# update, so we must include the `setVars` timing in this benchmarking comparison.
optVars_sde_benchmark = mesh_energy.NodalVars(m, 2)
sde_benchmark = dirichlet_demo.param_symdirichlet_element_ad(m, optVars_sde_benchmark)

benchmark.reset()
for i in range(200):
    optVars_sde_benchmark.setVars(optVars_sde_benchmark.getVars())
    sde_benchmark.gradient()
    sde_benchmark.hessian()
benchmark.report()

In [ ]:
optVars_param_sd_benchmark = mesh_energy.NodalVars(m, 2)
param_sd_benchmark = mesh_energy.Parametrization(m, optVars_param_sd_benchmark, energy.SymmetricDirichletDerivativeFree(2))

benchmark.reset()
for i in range(200):
    optVars_param_sd_benchmark.setVars(optVars_param_sd_benchmark.getVars())
    param_sd_benchmark.gradient()
    param_sd_benchmark.hessian()
benchmark.report()

In [ ]:
optVars_param_sd_an_benchmark = mesh_energy.NodalVars(m, 2)
param_sd_an_benchmark = mesh_energy.Parametrization(m, optVars_param_sd_an_benchmark, energy.SymmetricDirichlet(2))

benchmark.reset()
for i in range(200):
    optVars_param_sd_an_benchmark.setVars(optVars_param_sd_an_benchmark.getVars())
    param_sd_an_benchmark.gradient()
    param_sd_an_benchmark.hessian()
benchmark.report()